In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
df = pd.read_csv('cleaned_statements2.csv')
df.head(10)

,statement,status,label
0,oh gosh,Anxiety,0
1,trouble sleeping confused mind restless heart ...,Anxiety,0
2,wrong back dear forward doubt stay restless re...,Anxiety,0
3,shifted focus something else still worried,Anxiety,0
4,restless restless month boy mean,Anxiety,0
5,every break must nervous like something wrong ...,Anxiety,0
6,feel scared anxious may family protected,Anxiety,0
7,ever felt nervous know,Anxiety,0
8,slept well day like restless huh,Anxiety,0
9,really worried want cry,Anxiety,0


In [3]:
df.isnull().sum()

statement    0
status       0
label        0
dtype: int64

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

In [5]:
X = vectorizer.fit_transform(df['statement'])
y = df['label']

In [6]:
import pickle
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))

In [6]:
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

C:\Users\Admin\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [7]:
dataset = Dataset.from_pandas(df[['statement', 'label']])


In [8]:
# 2. Load tokenizer and model
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(df['label'].unique()))


C:\Users\Admin\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# 3. Tokenize
def tokenize_function(examples):
    return tokenizer(examples['statement'], padding='max_length', truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 52495/52495 [00:14<00:00, 3512.10 examples/s]


In [10]:


# 4. Train/test split
split = tokenized_datasets.train_test_split(test_size=0.2)
train_dataset = split['train']
eval_dataset = split['test']

In [11]:
from transformers import TrainingArguments

In [21]:
# 5. Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',       # This is now valid
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)


In [ ]:
# 6. Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# 7. Train
# trainer.train()

In [24]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=7)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Error while downloading from https://huggingface.co/distilbert-base-uncased/resolve/main/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


KeyboardInterrupt: 